In [2]:
!pip install feast scikit-learn 'feast[gcp]'

In [3]:
!feast version

Feast SDK Version: "0.64.0"


In [4]:
!git clone -b week_3 https://github.com/IITMBSMLOps/ga_resources.git

Cloning into 'ga_resources'...
remote: Enumerating objects: 81, done.
remote: Counting objects: 100% (81/81), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 81 (delta 14), reused 3 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (81/81), 1.38 MiB | 4.79 MiB/s, done.
Resolving deltas: 100% (14/14), done.


In [5]:
!mkdir -p /home/jupyter/iris_feast/data
!cp /home/jupyter/ga_resources/iris_data_adapted_for_feast.csv /home/jupyter/iris_feast/data/

In [6]:
!cd /home/jupyter/iris_feast && feast init iris_feature_repo


Creating a new Feast repository in /home/jupyter/iris_feast/iris_feature_repo.



In [21]:
import pandas as pd

# Read CSV
df = pd.read_csv("/home/jupyter/iris_feast/data/iris_data_adapted_for_feast.csv")

# Convert to Parquet
df.to_parquet("/home/jupyter/iris_feast/data/iris_data_adapted_for_feast.parquet")

print("Converted to Parquet!")
print(df.head())

Converted to Parquet!
              event_timestamp  iris_id  sepal_length  sepal_width  \
0  2025-09-17 10:40:17.102131     1001          5.52         2.53   
1  2025-09-18 10:40:17.102131     1001          5.50         2.24   
2  2025-09-19 10:40:17.102131     1001          5.55         2.47   
3  2025-09-20 10:40:17.102131     1001          5.45         2.37   
4  2025-09-21 10:40:17.102131     1001          5.65         2.52   

   petal_length  petal_width     species           created_timestamp  
0          3.86         1.13  versicolor  2025-10-02 10:40:17.172178  
1          3.60         1.08  versicolor  2025-10-02 10:40:17.172178  
2          3.75         1.08  versicolor  2025-10-02 10:40:17.172178  
3          3.92         1.20  versicolor  2025-10-02 10:40:17.172178  
4          3.95         1.17  versicolor  2025-10-02 10:40:17.172178  


In [29]:
import pandas as pd

# Read CSV
df = pd.read_csv("/home/jupyter/iris_feast/data/iris_data_adapted_for_feast.csv")

# Convert timestamps properly with timezone
df["event_timestamp"] = pd.to_datetime(df["event_timestamp"]).dt.tz_localize("UTC")
df["created_timestamp"] = pd.to_datetime(df["created_timestamp"]).dt.tz_localize("UTC")

# Save as parquet again
df.to_parquet("/home/jupyter/iris_feast/data/iris_data_adapted_for_feast.parquet")

print("Done! Fixed timestamps!")
print(df.dtypes)

Done! Fixed timestamps!
event_timestamp      datetime64[ns, UTC]
iris_id                            int64
sepal_length                     float64
sepal_width                      float64
petal_length                     float64
petal_width                      float64
species                           object
created_timestamp    datetime64[ns, UTC]
dtype: object


In [30]:
new_feature_def = """from datetime import timedelta
from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float64, String

iris_source = FileSource(
    path="/home/jupyter/iris_feast/data/iris_data_adapted_for_feast.parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created_timestamp",
)

iris_entity = Entity(
    name="iris_id",
    join_keys=["iris_id"],
)

iris_feature_view = FeatureView(
    name="iris_features",
    entities=[iris_entity],
    ttl=timedelta(days=365),
    schema=[
        Field(name="sepal_length", dtype=Float64),
        Field(name="sepal_width",  dtype=Float64),
        Field(name="petal_length", dtype=Float64),
        Field(name="petal_width",  dtype=Float64),
        Field(name="species",      dtype=String),
    ],
    source=iris_source,
)
"""

with open("/home/jupyter/iris_feast/iris_feature_repo/feature_repo/iris_features.py", "w") as f:
    f.write(new_feature_def)

print("Done!")

Done!


In [32]:
!cd /home/jupyter/iris_feast/iris_feature_repo/feature_repo && feast apply

/home/jupyter/iris_feast/iris_feature_repo/feature_repo/iris_features.py:11: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity 'iris_id'.
  iris_entity = Entity(
No project found in the repository. Using project name iris_feature_repo defined in feature_store.yaml
Applying changes for project iris_feature_repo
No changes to registry
No changes to infrastructure


In [33]:
import feast
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from joblib import dump

# Connect to Feast
fs = feast.FeatureStore(
    repo_path="/home/jupyter/iris_feast/iris_feature_repo/feature_repo"
)

# Get iris_id and timestamp from CSV
entity_df = pd.read_csv("/home/jupyter/iris_feast/data/iris_data_adapted_for_feast.csv")
entity_df["event_timestamp"] = pd.to_datetime(entity_df["event_timestamp"])
entity_df = entity_df[["iris_id", "event_timestamp"]]

# Ask Feast for features
training_df = fs.get_historical_features(
    entity_df=entity_df,
    features=[
        "iris_features:sepal_length",
        "iris_features:sepal_width",
        "iris_features:petal_length",
        "iris_features:petal_width",
        "iris_features:species",
    ],
).to_df()

print("----- Training Data from Feast -----")
print(training_df.head())

# Train model
le = LabelEncoder()
X = training_df[["sepal_length", "sepal_width", "petal_length", "petal_width"]]
y = le.fit_transform(training_df["species"])

model = RandomForestClassifier()
model.fit(X, y)

# Save model
dump(model, "/home/jupyter/iris_model.bin")
dump(le, "/home/jupyter/label_encoder.bin")
print("Model trained and saved!")

----- Training Data from Feast -----
   iris_id                  event_timestamp  sepal_length  sepal_width  \
0     1001 2025-09-17 10:40:17.102131+00:00          5.52         2.53   
1     1003 2025-09-17 10:40:17.102131+00:00          5.09         3.42   
2     1002 2025-09-17 10:40:17.102131+00:00          4.92         3.05   
3     1001 2025-09-18 10:40:17.102131+00:00          5.50         2.24   
4     1002 2025-09-18 10:40:17.102131+00:00          5.05         2.94   

   petal_length  petal_width     species  
0          3.86         1.13  versicolor  
1          1.34         0.22      setosa  
2          1.43         0.29      setosa  
3          3.60         1.08  versicolor  
4          1.53         0.31      setosa  
Model trained and saved!


In [34]:
!cd /home/jupyter/iris_feast/iris_feature_repo/feature_repo && feast materialize 2025-09-01T00:00:00 2025-10-03T00:00:00

/opt/micromamba/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=14237) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


Materializing 1 feature views from 2025-09-01 00:00:00+00:00 to 2025-10-03 00:00:00+00:00 into the sqlite online store.

iris_features:


In [35]:
!ls /home/jupyter/iris_feast/iris_feature_repo/feature_repo/data/

online_store.db  registry.db


In [36]:
import feast
import pandas as pd
from joblib import load

# Load saved model
model = load("/home/jupyter/iris_model.bin")
le = load("/home/jupyter/label_encoder.bin")

# Connect to Feast
fs = feast.FeatureStore(
    repo_path="/home/jupyter/iris_feast/iris_feature_repo/feature_repo"
)

# Give iris_id  Feast returns features from SQLite instantly
online_features = fs.get_online_features(
    entity_rows=[{"iris_id": 1001}],
    features=[
        "iris_features:sepal_length",
        "iris_features:sepal_width",
        "iris_features:petal_length",
        "iris_features:petal_width",
    ]
).to_dict()

print("Features from online store:", online_features)

# Make prediction
input_data = pd.DataFrame({
    "sepal_length": online_features["sepal_length"],
    "sepal_width":  online_features["sepal_width"],
    "petal_length": online_features["petal_length"],
    "petal_width":  online_features["petal_width"],
})

prediction = model.predict(input_data)
print("Predicted species:", le.inverse_transform(prediction))

Features from online store: {'iris_id': [1001], 'petal_width': [1.09], 'petal_length': [3.84], 'sepal_length': [5.45], 'sepal_width': [2.36]}
Predicted species: ['versicolor']


/opt/micromamba/lib/python3.12/site-packages/joblib/numpy_pickle.py:207: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  array.shape = self.shape


In [46]:
!bq mk --dataset project-b38b370e-25fb-420a-a54:feast_iris_store

Dataset 'project-b38b370e-25fb-420a-a54:feast_iris_store' successfully created.


In [47]:
new_yaml = """project: iris_feature_repo
registry: gs://mlops-iris-feast-pipeline-unique/registry.db
provider: gcp
online_store:
    type: datastore
offline_store:
    type: bigquery
    dataset: feast_iris_store
    project_id: project-b38b370e-25fb-420a-a54
entity_key_serialization_version: 3
auth:
    type: no_auth
"""

with open("/home/jupyter/iris_feast/iris_feature_repo/feature_repo/feature_store.yaml", "w") as f:
    f.write(new_yaml)

print("Done!")

Done!


In [55]:
from google.cloud import bigquery
import pandas as pd

# Read parquet file
df = pd.read_parquet("/home/jupyter/iris_feast/data/iris_data_adapted_for_feast.parquet")

# Upload to BigQuery
client = bigquery.Client(project="project-b38b370e-25fb-420a-a54")
table_id = "project-b38b370e-25fb-420a-a54.feast_iris_store.iris_features"

job = client.load_table_from_dataframe(df, table_id)
job.result()

print("Data uploaded to BigQuery!")
print(f"Table: {table_id}")

Data uploaded to BigQuery!
Table: project-b38b370e-25fb-420a-a54.feast_iris_store.iris_features


In [57]:
new_feature_def = """from datetime import timedelta
from feast import Entity, FeatureView, Field
from feast.infra.offline_stores.bigquery_source import BigQuerySource
from feast.types import Float64, String

# DATA SOURCE — now pointing to BigQuery!
iris_source = BigQuerySource(
    table="project-b38b370e-25fb-420a-a54.feast_iris_store.iris_features",
    timestamp_field="event_timestamp",
    created_timestamp_column="created_timestamp",
)

# ENTITY
iris_entity = Entity(
    name="iris_id",
    join_keys=["iris_id"],
)

# FEATURE VIEW
iris_feature_view = FeatureView(
    name="iris_features",
    entities=[iris_entity],
    ttl=timedelta(days=365),
    schema=[
        Field(name="sepal_length", dtype=Float64),
        Field(name="sepal_width",  dtype=Float64),
        Field(name="petal_length", dtype=Float64),
        Field(name="petal_width",  dtype=Float64),
        Field(name="species",      dtype=String),
    ],
    source=iris_source,
)
"""

with open("/home/jupyter/iris_feast/iris_feature_repo/feature_repo/iris_features.py", "w") as f:
    f.write(new_feature_def)

print("iris_features.py updated to use BigQuery!")

iris_features.py updated to use BigQuery!


In [58]:
!cd /home/jupyter/iris_feast/iris_feature_repo/feature_repo && feast apply

/opt/micromamba/lib/python3.12/site-packages/db_dtypes/core.py:51: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  _internal_fill_value = numpy.datetime64("NaT")
/home/jupyter/iris_feast/iris_feature_repo/feature_repo/iris_features.py:14: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity 'iris_id'.
  iris_entity = Entity(
No project found in the repository. Using project name iris_feature_repo defined in feature_store.yaml
Applying changes for project iris_feature_repo
Deploying infrastructure for iris_features


In [59]:
!cd /home/jupyter/iris_feast/iris_feature_repo/feature_repo && feast materialize 2025-09-01T00:00:00 2025-10-03T00:00:00

/opt/micromamba/lib/python3.12/site-packages/db_dtypes/core.py:51: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  _internal_fill_value = numpy.datetime64("NaT")
Materializing 1 feature views from 2025-09-01 00:00:00+00:00 to 2025-10-03 00:00:00+00:00 into the datastore online store.

iris_features:


In [60]:
import feast
import pandas as pd
from joblib import load

# Load saved model
model = load("/home/jupyter/iris_model.bin")
le = load("/home/jupyter/label_encoder.bin")

# Connect to Feast
fs = feast.FeatureStore(
    repo_path="/home/jupyter/iris_feast/iris_feature_repo/feature_repo"
)

# Get features from cloud Datastore (online store)
online_features = fs.get_online_features(
    entity_rows=[{"iris_id": 1001}],
    features=[
        "iris_features:sepal_length",
        "iris_features:sepal_width",
        "iris_features:petal_length",
        "iris_features:petal_width",
    ]
).to_dict()

print("Features from BigQuery online store:", online_features)

# Make prediction
input_data = pd.DataFrame({
    "sepal_length": online_features["sepal_length"],
    "sepal_width":  online_features["sepal_width"],
    "petal_length": online_features["petal_length"],
    "petal_width":  online_features["petal_width"],
})

prediction = model.predict(input_data)
print("Predicted species:", le.inverse_transform(prediction))

/opt/micromamba/lib/python3.12/site-packages/joblib/numpy_pickle.py:207: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  array.shape = self.shape


Features from BigQuery online store: {'iris_id': [1001], 'petal_width': [1.09], 'petal_length': [3.84], 'sepal_length': [5.45], 'sepal_width': [2.36]}
Predicted species: ['versicolor']
